[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-05-RegimeSwitching <<](./QC-Py-Cloud-05-RegimeSwitching.ipynb) | [Suivant : QC-Py-Cloud-06-VolTargeting >>](./QC-Py-Cloud-06-VolTargeting.ipynb)

# QC-Py-Cloud-06 — PCA Statistical Arbitrage Mean Reversion

**Module** : Hands-On AI Trading, Ch.6 — Applied Machine Learning, Example 13

**Objectif** : Implementer une strategie d'arbitrage statistique utilisant l'analyse en composantes principales (PCA) et la regression lineaire pour exploiter les ecarts de prix mean-reverting entre actions correlees.

**Approche cloud-native** : L'algorithme est execute sur QuantConnect Cloud. Les resultats sont presentes ci-dessous.

> **Note de conception** : Ce notebook est un descriptif pedagogique. Le code executable se trouve dans le projet QuantConnect Cloud correspondant (voir le lien dans le notebook). L'absence d'outputs est intentionnelle.


## 1. Concept : PCA Statistical Arbitrage

L'arbitrage statistique par PCA repose sur l'hypothese que les actions financieres sont entrainees par un petit nombre de facteurs communs (composantes principales). Quand une action s'ecarte significativement de la valeur predite par ces facteurs, elle a une probabilite elevee de revenir vers sa valeur attendue (mean reversion).

### Pipeline de la strategie

1. **Selection de l'univers** : Les 100 actions les plus liquides (dollar volume) avec prix > $5
2. **PCA** : Extraction des 3 premieres composantes principales des log-prix centres
3. **Regression OLS** : Pour chaque action, regression de son log-prix sur les facteurs PCA
4. **Residus** : Difference entre prix observe et prix predit par le modele
5. **Z-scores** : Standardisation des residus — les z-scores extremes signalent un ecart
6. **Trading** : Long sur les actions dont le z-score < -1.5 (ecart negatif = sous-evaluation)

### Pourquoi la PCA ?

Les marches financiers ont une structure factorielle : la majorite de la variance des prix peut etre expliquee par quelques facteurs (marche, secteur, taille). La PCA extrait ces facteurs de maniere non supervisee, sans necessiter de labels ou de connaissances prealables sur les secteurs.

### Tension pedagogique : stat-arb vs autres strategies

L'arbitrage statistique est fondamentalement different du trend following et du momentum :
- **Trend following** : acheter ce qui monte (momentum positif)
- **Mean reversion** : acheter ce qui a baisse (anticipation de rebond)
- **Stat-arb PCA** : acheter ce qui est sous-evalue par rapport aux facteurs communs

Le stat-arb PCA est plus sophistique que le mean reversion simple (RSI) car il utilise un modele factoriel multivarie plutot qu'un signal univarie.

## 2. Parametres de la strategie

| Parametre | Valeur | Justification |
|-----------|--------|---------------|
| `num_components` | 3 | Les 3 premieres composantes expliquent >90% de la variance |
| `lookback_days` | 60 | Environ 3 mois de donnees pour la PCA |
| `z_score_threshold` | 1.5 | Seuil de deviation (93.32e percentile) |
| `universe_size` | 100 | Top 100 actions les plus liquides |
| Rebalance | Mensuel | Recalcul PCA + poids chaque debut de mois |
| Capital initial | $1,000,000 | Taille suffisante pour un univers de 100 actions |

### Analyse des parametres (livre Broad)

- **num_components** : Minimum 2 (pour regression multiple), maximum 6 (composantes supplementaires n'apportent peu). Le Sharpe est maximal a 3 composantes.
- **lookback_days** : Minimum 21 (1 mois), maximum 126 (6 mois). Le Sharpe est maximise avec 126 jours de lookback.
- **z_score_threshold** : 1.5 est le seuil optimal identifie. Un seuil plus bas genere plus de trades mais moins selectif.

## 3. Resultats QC Cloud

**Projet QC Cloud** : 29463543 (`HandsOn-Ex13-PCA-StatArbitrage`)
**Periode** : 2019-01-01 au 2024-01-01 (5 ans)
**Capital initial** : $1,000,000

### Resultat principal (PCA-StatArb-v2-HandsOn)

| Metrique | Valeur |
|----------|--------|
| **Sharpe Ratio** | 0.399 |
| **CAGR** | 12.652% |
| **Max Drawdown** | 31.8% |
| **Net Profit** | +81.3% |
| **Total Orders** | 1,544 |
| **Win Rate** | 48% |
| **Loss Rate** | 52% |
| **Profit-Loss Ratio** | 1.30 |
| **Alpha** | 0.005 |
| **Beta** | 0.902 |
| **Sortino Ratio** | 0.476 |
| **Frais totaux** | $12,424.87 |
| **Portfolio Turnover** | 6.08% |

## 4. Analyse des performances

### 4.1 Le paradoxe du win rate

La strategie a un win rate de seulement 48% (moins d'une chance sur deux de gagner sur un trade individuel). Pourtant, elle genere un CAGR de 12.65% et un profit net de +81.3%. Comment est-ce possible ?

**Reponse** : Le profit-loss ratio de 1.30. Quand la strategie gagne, elle gagne en moyenne 0.80%. Quand elle perd, elle perd 0.61%. Ce ratio > 1 compense le win rate < 50%.

C'est une caracteristique commune des strategies mean-reversion : elles prennent beaucoup de petits paris sur des retours a la moyenne. La plupart echouent, mais les gains sur les trades reussis sont plus importants que les pertes.

### 4.2 Le drawdown eleve (31.8%)

Un MaxDD de 31.8% est eleve pour une strategie d'arbitrage. La cause principale : pendant les periodes de stress du marche (COVID-2020, 2022 bear), les relations factorielles entre actions se decomposent temporairement. Les actions qui etaient sous-evaluees par le modele PCA continuent de baisser.

Le beta de 0.902 confirme que la strategie reste exposee au risque marche : quand le marche baisse fortement, la strategie baisse aussi.

### 4.3 Alpha faible (0.005)

L'alpha annualise de 0.5% est marginal. La majorite du rendement (12.65%) vient de l'exposition au marche (beta 0.902), pas de la capacite du modele PCA a identifier des ecarts.

C'est une limitation fondamentale du stat-arb sur actions US liquides : les inefficiences de prix sont rapidement corrigees par les joueurs institutionnels, ne laissant que des traces d'alpha.

## 5. Comparaison avec les autres strategies Cloud

| Strategie | Sharpe | CAGR | MaxDD | Edge |
|-----------|--------|------|-------|------|
| Risk Parity v4 (Cloud-01) | 0.278 | 6.17% | 20.4% | Allocation diversifiee |
| Sector Rotation v3 (Cloud-02) | 0.614 | 10.76% | 15.5% | Trend following multi-asset |
| Dual Momentum v2 (Cloud-03) | 0.392 | 8.79% | 23.6% | Momentum + univers diversifie |
| Mean Reversion v2 (Cloud-04) | 0.307 | 5.89% | 14.6% | RSI + regime filter |
| **PCA Stat Arb (Cloud-06)** | **0.399** | **12.65%** | **31.8%** | **PCA + OLS residuals** |

### Observations

1. **CAGR le plus eleve** : 12.65%, superieur a toutes les autres strategies. Mais c'est un leurre — la majorite vient du beta, pas de l'alpha.

2. **Sharpe mediocre** : 0.399, inferieur au Sector Rotation (0.614) et comparable au Dual Momentum (0.392). Le ratio rendement/risque n'est pas exceptionnel.

3. **MaxDD catastrophique** : 31.8%, le plus eleve de toutes les strategies. En periode de stress, les positions stat-arb aggrave les pertes au lieu de les amortir.

4. **Sharpe-adjusted, le PCA stat-arb est inferieur au trend following** : Le Sector Rotation offre un Sharpe de 0.614 avec un MaxDD de 15.5%. Le PCA stat-arb offre un Sharpe de 0.399 avec un MaxDD de 31.8%. Le rendement ajuste au risque est nettement en faveur du trend following.

## 6. Algorithme detaille

Le code est deploye sur QC Cloud (projet 29463543). Le notebook local presente l'analyse des resultats, conformement au workflow cloud-native.

### Etape 1 : Selection de l'univers

```python
def _select_assets(self, fundamental):
    # Top 100 actions les plus liquides, prix > $5
    return [
        f.symbol
        for f in sorted(
            [f for f in fundamental if f.price > 5],
            key=lambda f: f.dollar_volume
        )[-self._universe_size:]
    ]
```

### Etape 2 : Calcul des poids via PCA + OLS

```python
def _get_weights(self, history):
    # 1. Log-transformer et centrer les prix
    sample = np.log(history.dropna(axis=1))
    sample -= sample.mean()

    # 2. Ajuster la PCA
    pca_model = PCA().fit(sample)

    # 3. Projeter sur les composantes principales
    factors = np.dot(sample, pca_model.components_.T)[:, :3]

    # 4. Regression lineaire pour chaque action
    residuals = {}
    for ticker in sample.columns:
        model = LinearRegression()
        model.fit(factors, sample[ticker])
        residuals[ticker] = sample[ticker] - model.predict(factors)

    # 5. Z-scores des residus
    resids = pd.DataFrame(residuals)
    zscores = ((resids - resids.mean()) / resids.std()).iloc[-1]

    # 6. Selectionner les actions sous-evaluees (z-score < -1.5)
    selected = zscores[zscores < -1.5]
    weights = selected * (1 / selected.abs().sum())
    return weights.sort_values()
```

### Etape 3 : Execution des positions

```python
# Poids negatifs inverses pour le mean reversion
self.set_holdings(
    [PortfolioTarget(symbol, -weight) for symbol, weight in weights.items()],
    True
)
```

**Note sur l'inversion des poids** : les z-scores selectionnes sont negatifs (< -1.5). Les poids sont proportionnels aux z-scores, donc negatifs. L'inversion (`-weight`) rend les positions longues. Plus le z-score est negatif, plus la position est importante.

## 7. Lecons principales

### 7.1 Le stat-arb PCA ne bat pas le trend following sur actions US

Malgre un CAGR eleve (12.65%), la strategie PCA stat-arb offre un ratio rendement/risque mediocre (Sharpe 0.399) et un drawdown catastrophique (31.8%). Le trend following multi-asset (Cloud-02) reste superieur avec un Sharpe de 0.614 et un MaxDD de 15.5%.

### 7.2 Le win rate n'est pas tout

Un win rate de 48% est compatible avec une strategie profitable si le profit-loss ratio est > 1. Le ratio de 1.30 compense les pertes frequentes par des gains plus importants.

### 7.3 Le regime de marche est critique

Les relations factorielles PCA se decomposent en periode de stress. Un filtre de regime (SMA200 ou VIX) pourrait reduire le drawdown, au prix d'un CAGR plus faible.

### 7.4 L'alpha est faible sur actions liquides

Sur un univers de 100 actions US liquides, les inefficiences sont rapidement exploitees. L'alpha de 0.5% confirme que le rendement provient principalement du beta. Des univers moins liquides (small caps, marches emergents) pourraient offrir plus d'alpha.

## 8. Pour aller plus loin

1. **Regime filter** : Ajouter un filtre VIX ou SMA200 pour reduire l'exposition en bear market. Comparer avec le Mean Reversion Cloud-04 ou le regime filter reduit le MaxDD de 42% a 15%.

2. **Ridge/LASSO regression** : Remplacer l'OLS par une regression regularisee pour reduire le surapprentissage sur les residus.

3. **Half-life weighting** : Utiliser la demi-vie du mean reversion (duree moyenne de retour a la moyenne) pour ajuster la taille des positions.

4. **Univers dynamique** : Augmenter l'univers a 200-300 actions pour capturer plus d'inefficiencies, ou reduire a 50 actions plus liquides pour diminuer les couts de transaction.

5. **PCA alternative : clustering sectoriel** : Remplacer la PCA non supervisee par un clustering sectoriel explicite (GICS) pour une interpretation plus claire des facteurs.

**Reference** : Avellaneda, M. & Lee, J.H. (2010) — "Statistical Arbitrage in the US Equities Market", Quantitative Finance. Broad, J. (2025) — *Hands-On AI Trading with Python, QuantConnect, and AWS*, Chapter 6, Example 13.